# Section 2 — Cyber-Attack Detection

This notebook demonstrates the complete NSL-KDD workflow:

- five-class attack mapping: Normal, DoS, Probe, R2L, and U2R;
- data profiling and class-imbalance analysis;
- leakage-safe numerical scaling and categorical encoding;
- ANOVA feature selection;
- class-balanced Random Forest;
- class-weighted one-dimensional CNN; and
- accuracy, macro metrics, confusion matrices, and precision-recall curves.

The official test partition remains untouched until final evaluation.

## 1. Locate the uploaded or local project

When using the VS Code Google Colab extension, upload these repository items together under `/content`:

- `pyproject.toml`
- `configs/`
- `scripts/`
- `src/`

Do not upload `.venv`, trained models, or raw datasets.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path


REPOSITORY_URL = "https://github.com/Saumyakeshi/ml_assignment.git"
COLAB_PROJECT_DIR = Path("/content/ml_assignment")


def is_project(path: Path) -> bool:
    return (path / "pyproject.toml").is_file() and (path / "src" / "comp70049").is_dir()


def find_local_project() -> Path | None:
    current = Path.cwd().resolve()
    return next((candidate for candidate in [current, *current.parents] if is_project(candidate)), None)


running_on_colab = Path("/content").is_dir()
if running_on_colab:
    if not is_project(COLAB_PROJECT_DIR):
        if COLAB_PROJECT_DIR.exists():
            raise FileExistsError(
                f"{COLAB_PROJECT_DIR} exists but is not a complete project. "
                "Remove or rename it, then rerun this cell."
            )
        subprocess.run(
            ["git", "clone", REPOSITORY_URL, str(COLAB_PROJECT_DIR)],
            check=True,
        )
    PROJECT_DIR = COLAB_PROJECT_DIR
else:
    PROJECT_DIR = find_local_project()
    if PROJECT_DIR is None:
        raise FileNotFoundError(
            "Could not find pyproject.toml and src/comp70049. "
            "Open the notebook from inside the repository."
        )

os.chdir(PROJECT_DIR)
SOURCE_DIR = PROJECT_DIR / "src"
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

print("Execution environment:", "Google Colab" if running_on_colab else "Local")
print("Project:", PROJECT_DIR)
print("Python:", sys.executable)

## 2. Install dependencies

The editable install makes the `comp70049` package importable and installs the optional PyTorch dependency used by the CNN.

In [ ]:
%pip install -q -e ".[deep]"

## 3. Imports and configuration

In [ ]:
import copy
import json
import subprocess

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

from comp70049.intrusion_detection.classic import (
    evaluate_random_forest,
    train_random_forest,
)
from comp70049.intrusion_detection.cnn import train_and_evaluate_cnn
from comp70049.intrusion_detection.data import (
    CLASS_NAMES,
    load_nsl_kdd,
    profile_partition,
)
from comp70049.intrusion_detection.evaluation import save_class_distribution_plot
from comp70049.intrusion_detection.preprocessing import (
    build_feature_pipeline,
    fit_transform_features,
    selected_feature_names,
)

config = json.loads(Path("configs/section_02.json").read_text(encoding="utf-8"))
config

## 4. Download and validate NSL-KDD

The original UNB page no longer hosts the files. The project downloader uses a public mirror and validates row counts, column counts, and SHA-256 checksums.

In [ ]:
subprocess.run(
    [sys.executable, "scripts/download_nsl_kdd.py"],
    check=True,
)

## 5. Load the official partitions and create validation data

Validation is split only from the official training data. The official test data is reserved for final evaluation.

In [ ]:
partitions = load_nsl_kdd(
    Path(config["data_dir"]),
    validation_fraction=float(config["validation_fraction"]),
    seed=int(config["seed"]),
)

profiles = {
    "train": profile_partition(partitions.train),
    "validation": profile_partition(partitions.validation),
    "test": profile_partition(partitions.test),
}

summary_rows = []
for split_name, profile in profiles.items():
    summary_rows.append({
        "split": split_name,
        "records": profile["records"],
        "missing_values": profile["missing_values"],
        "duplicate_records": profile["duplicate_records"],
        **profile["class_counts"],
    })

display(pd.DataFrame(summary_rows))

## 6. Inspect class imbalance

A logarithmic axis is necessary because U2R is extremely rare.

In [ ]:
training_counts = pd.Series(profiles["train"]["class_counts"])
axis = training_counts.plot.bar(
    color=["#4C78A8", "#E45756", "#F2CF5B", "#72B7B2", "#B279A2"],
    title="NSL-KDD training class distribution",
    ylabel="Records (log scale)",
    rot=0,
    logy=True,
)
axis.grid(axis="y", alpha=0.25)
plt.show()

display(training_counts.rename("records").to_frame())

## 7. Leakage-safe preprocessing and feature selection

The pipeline:

1. imputes and standardizes numerical features;
2. imputes and one-hot encodes categorical features;
3. removes zero-variance columns; and
4. selects 64 features using ANOVA F-values.

Every fitted transformation is learned from training data only.

In [ ]:
feature_pipeline = build_feature_pipeline(int(config["selected_features"]))
train_features, validation_features, test_features = fit_transform_features(
    feature_pipeline,
    partitions.train,
    partitions.validation,
    partitions.test,
)
feature_names = selected_feature_names(feature_pipeline)

print("Train matrix:", train_features.shape)
print("Validation matrix:", validation_features.shape)
print("Test matrix:", test_features.shape)
display(pd.Series(feature_names, name="selected_feature").to_frame())

In [ ]:
train_labels = partitions.train["label"].to_numpy(dtype=np.int64)
validation_labels = partitions.validation["label"].to_numpy(dtype=np.int64)
test_labels = partitions.test["label"].to_numpy(dtype=np.int64)

results_dir = Path(config["results_dir"])
models_dir = Path(config["models_dir"])
results_dir.mkdir(parents=True, exist_ok=True)
models_dir.mkdir(parents=True, exist_ok=True)

## 8. Train and evaluate the Random Forest baseline

In [ ]:
forest = train_random_forest(
    train_features,
    train_labels,
    config=config["random_forest"],
    seed=int(config["seed"]),
)
forest_metrics = evaluate_random_forest(
    forest,
    feature_pipeline,
    test_features,
    test_labels,
    class_names=CLASS_NAMES,
    selected_features=feature_names,
    model_dir=models_dir,
    results_dir=results_dir,
)

display(pd.DataFrame(forest_metrics["classification_report"]).T)
display(Image(filename=str(results_dir / "figures" / "random-forest-confusion-matrix.png")))
display(Image(filename=str(results_dir / "figures" / "random-forest-precision-recall-curves.png")))

## 9. Train and evaluate the 1D CNN

Keep `CNN_EPOCHS = 1` for a pipeline smoke test. Set it to `None` for the configured full run with early stopping.

The CNN treats the selected feature vector as a one-dimensional signal. This satisfies the requested CNN comparison, but the final report must acknowledge that tabular feature adjacency is not naturally spatial like image pixels.

In [ ]:
CNN_EPOCHS = 1  # Change to None for the configured full experiment.

cnn_config = copy.deepcopy(config["cnn"])
if CNN_EPOCHS is not None:
    cnn_config["epochs"] = CNN_EPOCHS

cnn_metrics = train_and_evaluate_cnn(
    train_features,
    train_labels,
    validation_features,
    validation_labels,
    test_features,
    test_labels,
    class_names=CLASS_NAMES,
    config=cnn_config,
    seed=int(config["seed"]),
    model_dir=models_dir,
    results_dir=results_dir,
)

display(pd.DataFrame(cnn_metrics["classification_report"]).T)
display(Image(filename=str(results_dir / "figures" / "1d-cnn-confusion-matrix.png")))
display(Image(filename=str(results_dir / "figures" / "1d-cnn-precision-recall-curves.png")))
display(Image(filename=str(results_dir / "figures" / "cnn-training-history.png")))

## 10. Direct model comparison

Macro measures give each class equal importance and therefore reveal failures hidden by overall accuracy.

In [ ]:
comparison = pd.DataFrame([
    {
        "model": "Random Forest",
        **{key: forest_metrics[key] for key in (
            "accuracy", "macro_precision", "macro_recall", "macro_f1",
            "weighted_f1", "macro_average_precision",
        )},
    },
    {
        "model": f"1D CNN ({cnn_metrics['epochs_completed']} epoch(s))",
        **{key: cnn_metrics[key] for key in (
            "accuracy", "macro_precision", "macro_recall", "macro_f1",
            "weighted_f1", "macro_average_precision",
        )},
    },
])
display(comparison.style.format(precision=4))
comparison.to_csv(results_dir / "model-comparison.csv", index=False)

## 11. Analysis prompts

Use the saved outputs to answer these questions in the report:

1. Why does accuracy give an incomplete picture?
2. Which classes are most often misclassified as normal?
3. Why are R2L and U2R difficult to learn?
4. Does class weighting improve minority recall at an unacceptable precision cost?
5. How does the presence of test-only attack types affect generalization?
6. Does the CNN provide enough benefit to justify its computation?
7. What does the age of NSL-KDD imply for real deployment?

Do not use the one-epoch CNN result as the final experiment.

## 12. Optional: download the generated results from Colab

In [ ]:
import shutil

archive = shutil.make_archive(
    "/content/section_02_results",
    "zip",
    root_dir=results_dir,
)
print("Download this file from the Colab Contents panel:", archive)